# Motion Correction and Gaussian Smoothing

**NX-421 - Mini-project, Part 1 (2a and 2b)**

In this part, we correct for head motion in the language-task fMRI run and then apply Gaussian smoothing. We also calculate the framewise displacement to look at how much the subject moved between consecutive volumes.

We use MCFLIRT and `fslmaths`, as seen in Lab 3. The resulting images and motion parameters will be used in the next part for alignment checks and the GLM.

The notebook is set up for Neurodesk, with the dataset in `data/` and the repository root as the working directory.

## 0. Setup

First, we load FSL and the Python libraries. We use the same FSL version as in the labs. The paths below correspond to our project folder; `DATA_DIR` can be changed if the data are stored somewhere else.

In [ ]:
import os
import sys
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt
from nilearn import image, plotting

# Load FSL in Neurodesk, as in Labs 2 and 3.
import module as lmod
await lmod.purge(force=True)
await lmod.load('fsl/6.0.7.18')

from fsl.wrappers import mcflirt

# Keep the output extension consistent with the downstream notebooks.
os.environ['FSLOUTPUTTYPE'] = 'NIFTI_GZ'
if not os.environ.get('FSLDIR'):
    raise RuntimeError('FSLDIR is not set. Check that the Neurodesk FSL module loaded.')
for tool in ['mcflirt', 'fslmaths']:
    if shutil.which(tool) is None:
        raise RuntimeError(f'{tool} is unavailable. Check the Neurodesk FSL module.')

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__, '| pandas:', pd.__version__, '| NiBabel:', nib.__version__)
fsl_version_path = Path(os.environ['FSLDIR']) / 'etc' / 'fslversion'
if fsl_version_path.is_file():
    print('FSL:', fsl_version_path.read_text().strip())

In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
FUNC_DATA_DIR = DATA_DIR / 'fMRI' / 'tfMRI_LANGUAGE_LR'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'functional'

raw_fmri_path = FUNC_DATA_DIR / 'tfMRI_LANGUAGE_LR.nii.gz'
# MCFLIRT takes an output basename, without .nii.gz.
moco_basename = OUTPUT_DIR / 'tfMRI_LANGUAGE_LR_mcf'
moco_fmri_path = OUTPUT_DIR / 'tfMRI_LANGUAGE_LR_mcf.nii.gz'
motion_par_path = OUTPUT_DIR / 'tfMRI_LANGUAGE_LR_mcf.par'
smoothed_fmri_path = OUTPUT_DIR / 'tfMRI_LANGUAGE_LR_mcf_smooth.nii.gz'

FD_THRESHOLD = 0.2  # mm, inspection threshold used in Lab 3
FWHM_MM = 6.0       # same smoothing width as Lab 3

if not raw_fmri_path.is_file():
    raise FileNotFoundError(f'Input not found: {raw_fmri_path}. Check DATA_DIR and the working directory.')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Input:', raw_fmri_path)
print('Outputs:', OUTPUT_DIR)

description_path = DATA_DIR / 'dataset_description.md'
if description_path.is_file():
    print('Dataset description:\n', description_path.read_text())
else:
    print('dataset_description.md was not found under DATA_DIR; inspect the supplied description separately.')

### 0.1 Looking at the input data

Before processing the images, we check their dimensions, voxel sizes and TR. The dataset description is also useful to find out which preprocessing steps have already been applied. If the supplied images have already been realigned, the motion parameters estimated here will only describe the remaining movement.

Like in Lab 3, we plot the mean intensity over time to look for unusual volumes, especially at the start of the run. Changes in these first volumes can come from magnetisation not yet reaching a steady state, rather than from head movement.

We keep all volumes so that the timing stays consistent with the events used for the GLM. Suspicious volumes can then be accounted for in that analysis.

In [ ]:
raw_img = nib.load(str(raw_fmri_path))
if len(raw_img.shape) != 4 or raw_img.shape[3] < 2:
    raise ValueError('Expected a 4D fMRI run with at least two volumes.')

n_volumes = raw_img.shape[3]
voxel_sizes = raw_img.header.get_zooms()[:3]
tr_header = float(raw_img.header.get_zooms()[3])
print('Dimensions:', raw_img.shape)
print('Voxel sizes:', voxel_sizes)
print('Spatial/time units:', raw_img.header.get_xyzt_units())
print('TR (header units):', tr_header)
print('Orientation:', nib.aff2axcodes(raw_img.affine))
print('Affine:\n', raw_img.affine)

# Same whole-volume mean as Lab 3; load one volume at a time to limit memory use.
mean_intensity = np.array([
    np.asarray(raw_img.dataobj[..., v], dtype=np.float32).mean()
    for v in range(n_volumes)
])
if not np.all(np.isfinite(mean_intensity)):
    raise ValueError('Non-finite values found in the input mean-intensity trace.')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.arange(n_volumes), mean_intensity)
ax.set(xlabel='Volume index (0-based)', ylabel='Mean voxel intensity',
       title='Supplied run: whole-volume mean intensity')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'mean_intensity.png', dpi=200)
plt.show()

## 1. Motion Correction (2a)

### 1.1 MCFLIRT

When the subject moves, a voxel may no longer correspond to the same part of the brain across the run. To reduce this problem, we use MCFLIRT, to align the volumes to a common reference.

We keep the default reference, which is the middle volume, and use six degrees of freedom: three rotations and three translations. This allows rigid-body movement without changing the shape or size of the brain.

The `.par` file stores the motion parameters for each volume, with rotations in radians and translations in millimetres. We also save the transformation matrices using `mats=True`.

In [ ]:
mcflirt(
    infile=str(raw_fmri_path), o=str(moco_basename),
    plots=True, report=True, dof=6, mats=True
)

for path in [moco_fmri_path, motion_par_path]:
    if not path.is_file():
        raise FileNotFoundError(f'MCFLIRT output missing: {path}')

moco_img = nib.load(str(moco_fmri_path))
assert moco_img.shape == raw_img.shape, 'MCFLIRT changed the image dimensions.'
assert np.allclose(moco_img.affine, raw_img.affine), 'Unexpected change in the spatial grid.'
assert np.allclose(moco_img.header.get_zooms(), raw_img.header.get_zooms()), 'Voxel sizes or TR changed.'

matrix_dir = Path(str(moco_basename) + '.mat')
matrix_paths = sorted(matrix_dir.glob('MAT_*'))
assert len(matrix_paths) == n_volumes, 'Expected one MCFLIRT matrix per volume.'
for path in matrix_paths:
    matrix = np.loadtxt(path)
    assert matrix.shape == (4, 4) and np.all(np.isfinite(matrix)), f'Invalid matrix: {path}'
print('MCFLIRT outputs and geometry checked.')

### 1.2 Motion parameters

We load the `.par` file with the function from Lab 3. The only small change is the whitespace separator, so that it also works if the spacing between columns varies. We then plot rotations and translations separately because they have different units.

In [ ]:
def load_mot_params_fsl_6_dof(path):
    return pd.read_csv(path, sep=r'\s+', header=None,
            engine='python', names=['Rotation x', 'Rotation y', 'Rotation z',
                                    'Translation x', 'Translation y', 'Translation z'])

mot_params = load_mot_params_fsl_6_dof(motion_par_path)
assert mot_params.shape == (n_volumes, 6), 'Motion parameters must have one row per volume and six columns.'
assert np.all(np.isfinite(mot_params.to_numpy())), 'Invalid motion parameters.'
mot_params.head()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for name in ['Rotation x', 'Rotation y', 'Rotation z']:
    axes[0].plot(mot_params.index, mot_params[name], label=name)
for name in ['Translation x', 'Translation y', 'Translation z']:
    axes[1].plot(mot_params.index, mot_params[name], label=name)
axes[0].set(ylabel='Rotation (radians)', title='MCFLIRT motion parameters relative to the reference')
axes[1].set(xlabel='Volume index (0-based)', ylabel='Translation (mm)')
for ax in axes:
    ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'motion_parameters.png', dpi=200)
plt.show()

### 1.3 Framewise displacement

The motion parameters describe the position relative to the reference volume. To look at movement between consecutive volumes, we calculate framewise displacement (FD) with `compute_FD_power`.

The function adds the absolute differences in translation and rotation. Rotations are converted to distances using a sphere with a radius of 50 mm:

$$FD_t = |\Delta x_t| + |\Delta y_t| + |\Delta z_t| + 50(|\Delta\alpha_t| + |\Delta\beta_t| + |\Delta\gamma_t|).$$

There is no FD value for the first volume because it has no previous volume to compare with. We therefore get N - 1 values, starting at volume index 1 (indices start at 0).

We use the same 0.2 mm threshold as in the lab to highlight frames worth inspecting. We do not remove them automatically. The mean and maximum FD, along with the flagged indices, are printed below.

In [ ]:
def compute_FD_power(mot_params):
    framewise_diff = mot_params.diff().iloc[1:]

    rot_params = framewise_diff[['Rotation x', 'Rotation y', 'Rotation z']]
    # Estimating displacement on a 50 mm radius sphere, as in Lab 3.
    converted_rots = rot_params*50
    trans_params = framewise_diff[['Translation x', 'Translation y', 'Translation z']]
    fd = converted_rots.abs().sum(axis=1) + trans_params.abs().sum(axis=1)
    return fd

fd = compute_FD_power(mot_params).to_numpy()
fd_volume_indices = np.arange(1, n_volumes)
assert fd.shape == (n_volumes - 1,) and np.all(np.isfinite(fd))
assert np.all(fd >= 0)
flagged_volumes = fd_volume_indices[fd > FD_THRESHOLD]

pd.DataFrame({
    'volume_index': fd_volume_indices,
    'fd_mm': fd,
    'above_threshold': fd > FD_THRESHOLD
}).to_csv(OUTPUT_DIR / 'framewise_displacement.csv', index=False)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(fd_volume_indices, fd)
ax.axhline(FD_THRESHOLD, color='black', linestyle='--', label=f'FD threshold: {FD_THRESHOLD} mm')
ax.set(xlabel='Volume index (0-based; first FD at volume 1)', ylabel='FD (mm)',
       title='Framewise displacement from MCFLIRT parameters')
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'framewise_displacement.png', dpi=200)
plt.show()

print(f'Mean FD (N - 1 transitions): {fd.mean():.4f} mm')
print(f'Maximum FD: {fd.max():.4f} mm at volume {fd_volume_indices[np.argmax(fd)]}')
print(f'Transitions above {FD_THRESHOLD} mm: {len(flagged_volumes)}/{fd.size} ({100 * len(flagged_volumes) / fd.size:.1f}%)')
print('Flagged volume indices (0-based):', flagged_volumes.tolist())
print('All volumes retained; pass these indices to the downstream GLM analysis.')

### 1.4 Checking the correction

We compare the original and corrected images at the volume with the highest FD, using the same slices and intensity scale. We can also open both runs in FSLeyes to go through the volumes and check whether the brain stays better aligned over time.

The FD calculated above describes the estimated movement in the input run; it does not measure how much movement remains after correction. As discussed in Lab 3, realignment cannot correct every signal change caused by motion. This is why the motion parameters are also useful as nuisance regressors in the GLM.

In [ ]:
qc_volume_index = int(fd_volume_indices[np.argmax(fd)])
raw_volume = image.index_img(raw_img, qc_volume_index)
corrected_volume = image.index_img(moco_img, qc_volume_index)
raw_volume_data = raw_volume.get_fdata()
corrected_volume_data = corrected_volume.get_fdata()
assert np.all(np.isfinite(corrected_volume_data)), 'Non-finite corrected volume.'

# A common physical coordinate in both images, at the centre of the input grid.
centre_voxel = (np.array(raw_img.shape[:3]) - 1) / 2
cut_coords = tuple(nib.affines.apply_affine(raw_img.affine, centre_voxel))
vmin = min(raw_volume_data.min(), corrected_volume_data.min())
vmax = max(raw_volume_data.max(), corrected_volume_data.max())

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
for ax, volume, title in zip(axes, [raw_volume, corrected_volume],
                             ['Supplied run', 'Motion corrected']):
    plotting.plot_epi(volume, axes=ax, display_mode='ortho', cut_coords=cut_coords,
                      cmap='gray', vmin=vmin, vmax=vmax, colorbar=True,
                      title=f'{title}: volume {qc_volume_index}')
fig.savefig(OUTPUT_DIR / 'motion_correction_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Optional interactive QC, as in the labs. Set True on the VM to launch FSLeyes.
OPEN_FSLEYES = False
if OPEN_FSLEYES:
    subprocess.Popen(['fsleyes', str(raw_fmri_path), str(moco_fmri_path)])

**Results to discuss for after:** Are the FD peaks isolated or spread over several consecutive volumes? Do the corresponding frames look better aligned after correction? The mean and maximum FD give a summary of the run, but the flagged frames also need a visual check. Any unusual intensity values at the start should be considered separately.

## 2. Gaussian Smoothing (2b)

### 2.1 Applying the filter

Gaussian smoothing averages nearby voxels, giving more weight to those closest to the centre of the kernel. As seen in Lab 3, this can reduce noise, but it also blurs small structures and fine activation patterns.

We use a FWHM of 6 mm, following the example in the lab. This gives a reasonable starting point for spatial averaging, although it is not a value optimised for this particular subject. The code also prints its size relative to the voxel dimensions.

FSL takes the Gaussian standard deviation in millimetres, so we first convert FWHM to sigma:

$$\sigma = \frac{FWHM}{2.3548}.$$

We apply this filter to the motion-corrected run with `fslmaths -s`. Smoothing is spatial only; we do not smooth across time.

In [ ]:
sigma_mm = FWHM_MM / 2.3548
print(f'FWHM: {FWHM_MM:g} mm; sigma: {sigma_mm:.4f} mm')
print('FWHM / voxel size along each image axis:', FWHM_MM / np.array(voxel_sizes))

# Lab 3 smoothing command, with project paths and explicit failure checking.
subprocess.run([
    'fslmaths', str(moco_fmri_path), '-s', str(sigma_mm), str(smoothed_fmri_path)
], check=True)

if not smoothed_fmri_path.is_file():
    raise FileNotFoundError(f'Smoothing output missing: {smoothed_fmri_path}')
smoothed_img = nib.load(str(smoothed_fmri_path))
assert smoothed_img.shape == moco_img.shape, 'Smoothing changed the image dimensions.'
assert np.allclose(smoothed_img.affine, moco_img.affine), 'Smoothing changed the spatial grid.'
assert np.allclose(smoothed_img.header.get_zooms(), moco_img.header.get_zooms()), 'Voxel sizes or TR changed.'
print('Smoothed image geometry checked.')

### 2.2 Before and after smoothing

We choose the middle volume to show the effect of smoothing. Both images have already been motion-corrected. We keep the same slices, colour map and intensity limits in the two plots to make the comparison easier.

In [ ]:
volume_index = n_volumes // 2
before_smoothing = image.index_img(moco_img, volume_index)
after_smoothing = image.index_img(smoothed_img, volume_index)
before_data = before_smoothing.get_fdata()
after_data = after_smoothing.get_fdata()
assert np.all(np.isfinite(before_data)) and np.all(np.isfinite(after_data)), 'Invalid comparison volume.'
vmin = min(before_data.min(), after_data.min())
vmax = max(before_data.max(), after_data.max())

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
for ax, volume, title in zip(
    axes, [before_smoothing, after_smoothing],
    ['Before smoothing (motion corrected)', f'After smoothing ({FWHM_MM:g} mm FWHM)']
):
    plotting.plot_epi(volume, axes=ax, display_mode='ortho', cut_coords=cut_coords,
                      cmap='gray', vmin=vmin, vmax=vmax, colorbar=True,
                      title=f'{title}: volume {volume_index}')
fig.savefig(OUTPUT_DIR / 'smoothing_comparison.png', dpi=200, bbox_inches='tight')
plt.show()
print('Compared volume index (0-based):', volume_index)
print('Slice coordinates (image world space):', cut_coords)

**Results to discuss for after:** The main things to look at are the image texture and the sharpness of brain boundaries. We expect a smoother image, with some loss of fine detail. This visual comparison alone cannot tell us whether task detection will improve in the GLM.

## 3. Saved outputs

The following files are saved in `outputs/functional/` for the next part:

- `tfMRI_LANGUAGE_LR_mcf.nii.gz`: motion-corrected run, before smoothing.
- `tfMRI_LANGUAGE_LR_mcf.par`: the six motion parameters for each volume.
- `tfMRI_LANGUAGE_LR_mcf_smooth.nii.gz`: motion-corrected and smoothed run.

The transformation matrices, FD values and figures are saved there as well. The FD CSV starts at volume index 1, since the first volume has no preceding frame. The GLM notebook adds an initial zero when it needs one value per volume.

We keep the original number of volumes and event timing. The next notebook handles anatomical alignment and GLM design, using these outputs. Since the final image is already smoothed, this step does not need to be repeated there.

In [ ]:
expected_outputs = [
    moco_fmri_path, motion_par_path, smoothed_fmri_path,
    OUTPUT_DIR / 'framewise_displacement.csv',
    OUTPUT_DIR / 'mean_intensity.png',
    OUTPUT_DIR / 'motion_parameters.png',
    OUTPUT_DIR / 'framewise_displacement.png',
    OUTPUT_DIR / 'motion_correction_comparison.png',
    OUTPUT_DIR / 'smoothing_comparison.png',
]
for path in expected_outputs:
    if not path.is_file():
        raise FileNotFoundError(f'Missing output: {path}')
    print('[OK]', path.relative_to(PROJECT_ROOT))

print(f'Volumes in supplied / corrected / smoothed runs: {n_volumes} / {moco_img.shape[3]} / {smoothed_img.shape[3]}')
print('Volumes removed: 0')
print('Automated checks passed. Complete the visual inspection and interpretation before reporting results.')